In [37]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import sklearn.metrics as skm
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import gensim.downloader as api
from sklearn.decomposition import PCA

import spacy
import re
import warnings

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
import joblib
import os

In [38]:
df = pd.read_csv("../data/preprocessed/reviews_sentiment_analysis.csv")

In [39]:
param_grid = {
    'hidden_layer_sizes': [(16), (8), (16, 8), (8,4)],
    'activation': ['logistic', 'tanh', 'relu'],
    'solver': ['lbfgs'],
    'alpha': [0.1, 1.0, 10.0],
    'early_stopping': [True],
    'batch_size': [4, 8, 16],
    'n_iter_no_change': [10, 20]
}

In [40]:
"""
  Load target features on y and drop target features on X.
  Perform shuffle on first split to prevent bias. Split 80-20 for train-test.
  Stratifies all subsets to ensure equal samples for each class.
"""
def train_test_valid_split(df, targets):
  y = df[targets]
  X = df.drop(targets, axis=1)

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state=101, shuffle=True, stratify=y)

  return X_train, X_test, y_train, y_test

In [41]:
parts_speech = spacy.load("en_core_web_sm")
"""
    Performs preprocessing of the text for GloVe vectorization.
    Lowercases and tokenizes the entire text while removing stopwords and punctuation performing lemmatization.
    Then loads in a pretrained GloVe model and performs mean pooling to get the average of the sentence vector embeddings for each text.
    Only lemmas that are in the pre-trained GloVe vocabulary are accounted for computation.
    Returns the average of the sentence vector embeddings.
"""
def preprocess(text, w2v_model):
    text_pos = parts_speech(text.lower())
    vectors = []

    for token in text_pos:
        if not token.is_stop and not token.is_punct and token.lemma_ in w2v_model:
            vectors.append(w2v_model[token.lemma_])

    return np.mean(vectors, axis=0)

In [42]:
# Load in pre-trained GloVe Model
pretrain_w2v_model = api.load("glove-wiki-gigaword-100")

In [43]:
# Performs preprocessing
df['Review'] = df.apply(lambda row: preprocess(row['Review'], pretrain_w2v_model), axis=1)

In [ ]:
"""
    Converts vectors to numpy array to allow the scaler to scale the vectors.
    Then performs scaling on training and test input vectors. 
    Principal Component Analysis is performed to do dimensionality reduction to reduce number of the embeddings to 20.
    Returns scaled input vectors of training and test.
"""
def scale_embeddings(X_train, X_test):
    X_train_mat = np.array(X_train['Review'].tolist())
    X_test_mat = np.array(X_test['Review'].tolist())

    scaler = StandardScaler()
    X_train_mat = scaler.fit_transform(X_train_mat)
    X_test_mat = scaler.transform(X_test_mat)

    pca = PCA(n_components=20)
    X_train_mat = pca.fit_transform(X_train_mat)
    X_test_mat = pca.transform(X_test_mat)    

    return X_train_mat, X_test_mat

In [45]:
"""
    Helper method to saves the best model as joblib file.
"""
def save_model(model):
    best_model = model.best_estimator_

    model_fp = os.path.join("../models", "best_mlp_model.joblib")

    joblib.dump(best_model, model_fp)

    print(f"Saved best MLP Classifer to {model_fp}")

    return best_model    

In [ ]:
"""
    Helper method to print classification reports as well as the training and test accuracy and f1 score.
"""
def printResults(classifer, X_train, X_test, y_train, y_test):
    train_pred = classifer.predict(X_train)
    test_pred = classifer.predict(X_test)

    print("Training Classification Report:\n", classification_report(y_train, train_pred))

    train_report = classification_report(y_train, train_pred, output_dict=True)
    print(f"Training Accuracy {train_report['accuracy']:.4f}")
    print(f"Training F1-Score: {train_report['macro avg']['f1-score']:.4f}\n") 

    print("Test Classification Report:\n", classification_report(y_test, test_pred))

    test_report = classification_report(y_test, test_pred, output_dict=True)
    print(f"Test Accuracy {test_report['accuracy']:.4f}")
    print(f"Test F1-Score: {test_report['macro avg']['f1-score']:.4f}\n") 

In [47]:
"""
    Performs binary sentiment classification with MLP.
    If param_grid is given, then run grid search to find the best parameters.
    Once the best parameters are found, then retrain the classifier with best parameters and save it.
    Then print out the training and test results. 
"""
def mlp_sentiment_classifer(param_grid=None):
    if not param_grid:
        return

    X_train, X_test, y_train, y_test = train_test_valid_split(df, "positive=1/negative=0")
    X_train, X_test = scale_embeddings(X_train, X_test)

    mlp = GridSearchCV(MLPClassifier(max_iter=1000, random_state=101), param_grid, cv=5, scoring="f1")
    mlp.fit(X_train, y_train)

    print(f'Best Model Parameters: {mlp.best_params_}')
    print(f'Best Model Score: {mlp.best_score_}')
    mlp_best_model = save_model(mlp)

    printResults(mlp_best_model, X_train, X_test, y_train, y_test)

In [48]:
mlp_sentiment_classifer(param_grid)

Best Model Parameters: {'activation': 'relu', 'alpha': 10.0, 'batch_size': 4, 'early_stopping': True, 'hidden_layer_sizes': 16, 'n_iter_no_change': 10, 'solver': 'lbfgs'}
Best Model Score: 0.8630865056749789
Saved best MLP Classifer to ../models/best_mlp_model.joblib
Training Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.93      0.93       100
           1       0.94      0.94      0.94       108

    accuracy                           0.94       208
   macro avg       0.94      0.94      0.94       208
weighted avg       0.94      0.94      0.94       208

Training Accuracy 0.9375
Training F1-Score: 0.9374

Test Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.76      0.79        25
           1       0.79      0.85      0.82        27

    accuracy                           0.81        52
   macro avg       0.81      0.81      0.81        52
weighted avg       